# Baseline agent (Strategy 0 - no verification)

> User request -> extract constraints -> retrieve info -> plan itinerary -> return

Done: State, schemas, graph and run()

TBC: The LLM nodes + tools

## 0. Setup

In [1]:
# %pip install python-dotenv pandas pydantic
import os, json, time
from pathlib import Path
from typing import Any, Literal, Optional, TypedDict
from urllib.request import Request, urlopen
from urllib.error import HTTPError

import pandas as pd
from pydantic import BaseModel, Field

from dotenv import load_dotenv
load_dotenv() # read .env

URL = os.environ.get('OPENROUTER_BASE_URL')
KEY = os.environ.get('OPENROUTER_API_KEY')

MODEL = os.environ.get('PLANNER_MODEL', 'openai/gpt-4.1')
TEMPERATURE = float(os.environ.get('PLANNER_TEMPERATURE', '0.7'))
# look for data/processed starting from here and going up folder by folder
DATA = None
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / 'data' / 'processed').exists():
        DATA = folder / 'data' / 'processed'
        break
print(MODEL, TEMPERATURE, DATA)

openai/gpt-4.1 0.7 /Users/enruifung/Desktop/DSA4213/Group Project/Agentic-Travel-Planner-Ablation-of-Verifications-/data/processed


## 1. Schemas

Multiple Constraints, Itinerary -> Day -> Activity

In [2]:
# Verification State
VState = Literal['UNRESOLVED', 'SATISFIED', 'VIOLATED', 'UNVERIFIABLE']

class Constraint(BaseModel):
    type: str # type of constraint (e.g. "budget", "diet")
    value: str
    is_hard: bool = True # whether constraint is hard (must follow) / soft (nice to have)
    state: VState = 'UNRESOLVED'
    note: Optional[str] = None # if a constraint is ambiguous

class Activity(BaseModel):
    name: str
    category: Optional[str] = None
    start_time: Optional[str] = None
    end_time: Optional[str] = None
    location: Optional[str] = None
    price_usd: Optional[float] = None

class Day(BaseModel):
    day_index: int
    activities: list[Activity] = Field(default_factory=list) # a list of activity in a day

# The whole plan
class Itinerary(BaseModel):
    destination: Optional[str] = None # currently only New York
    days: list[Day] = Field(default_factory=list) # a list of different days
    total_cost_usd: Optional[float] = None
    summary: Optional[str] = None # short summary of itinerary

## 2. State + Log

In [3]:
# A dictionary that gets passed from node to node
class State(TypedDict, total=False):
    # each node gets:
    request: str # user's original text
    constraints: list[Constraint]
    info: dict[str, Any] # filled in by the retrieve node
    draft: Optional[Itinerary]
    final: Optional[Itinerary]

# Log down what the agent did
LOG = []

def log(kind, **d):
    LOG.append({'kind': kind, 't': round(time.time(), 3), **d})

## 3. Model call

In [4]:
# Sends a chat request to OpenRouter & returns the model's reply as text
def ask(messages, temperature=TEMPERATURE):
    if not KEY:
        raise RuntimeError('Set OPENROUTER_API_KEY in .env')
    body = {'model': MODEL, 'messages': messages, 'temperature': temperature}
    req = Request(URL + '/chat/completions',
                  data=json.dumps(body).encode(),
                  headers={'Authorization': 'Bearer ' + KEY, 'Content-Type': 'application/json'},
                  method='POST')
    try:
        with urlopen(req, timeout=90) as r:
            raw = json.load(r)
    except HTTPError as e:
        raise RuntimeError(f'request failed (HTTP {e.code}); check model/key/credit') from None
    return raw['choices'][0]['message']['content']

# Returns in JSON and turn into a Python object (useful for extract nodes & plan nodes)
def ask_json(messages, tries=3):
    for i in range(tries):
        text = ask(messages).strip()
        if text.startswith('```'):
            text = text.split('```')[1].removeprefix('json').strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            log('bad_json', attempt=i + 1)
            messages = messages + [{'role': 'user', 'content': 'Reply with valid JSON only.'}]
    raise ValueError('no valid JSON after retries')

## 4. Tools (TBC)

Read `data/processed/` only.

In [5]:
attractions = pd.read_csv(DATA / 'attractions.csv')
restaurants = pd.read_csv(DATA / 'restaurants.csv')
transport = pd.read_csv(DATA / 'transport_modes.csv')
hotels = pd.read_csv(DATA / 'accomodation.csv')
print(len(attractions), len(restaurants), len(hotels))

3066 670 365


### Tool 1: search_attractions

In [6]:
def search_attractions(category=None, borough=None, limit=10):
    df = attractions
    # filter according to category (if have)
    if category:
        df = df[df['category'].str.contains(category, case=False, na=False)]
    # borough: area of the city (in attractions.csv)
    if borough:
        df = df[df['borough'].str.contains(borough, case=False, na=False)]
    cols = ['attraction_id', 'name', 'category', 'borough', 'admission_price_usd']
    # turn into dict
    rows = df[cols].head(limit).to_dict('records')
    # rmb to log
    log('tool', name='search_attractions', n=len(rows))
    return rows

search_attractions(category='park', limit=3)

[{'attraction_id': 'ATTR_0002',
  'name': 'HARLEM RIVER PARK W 145 ST',
  'category': 'park',
  'borough': 'Manhattan',
  'admission_price_usd': nan},
 {'attraction_id': 'ATTR_0005',
  'name': 'HARLEM RIVER PARK LEXINGTON AVE & E 130',
  'category': 'park',
  'borough': 'Manhattan',
  'admission_price_usd': nan},
 {'attraction_id': 'ATTR_0008',
  'name': 'PROSPECT PARK BOAT HOUSE',
  'category': 'park',
  'borough': 'Brooklyn',
  'admission_price_usd': nan}]

In [7]:
# (TBC)
def search_restaurants(dietary=None, borough=None, limit=10):
    log('tool', name='search_restaurants'); return []

def search_accommodation(borough=None, max_price=None, limit=10):
    log('tool', name='search_accommodation'); return []

def get_travel_time(origin, destination):
    log('tool', name='get_travel_time'); return None

def get_travel_cost(origin, destination, mode='subway'):
    # cost = base_fare + time * rate_per_km, from transport_modes.csv
    log('tool', name='get_travel_cost'); return None

def calculate_total_cost(items):
    log('tool', name='calculate_total_cost'); return None

def get_weather(date, borough):
    log('tool', name='get_weather'); return None

def get_ticket_availability(activity, date, time, travellers):
    log('tool', name='get_ticket_availability'); return None  # no ticket data yet

## 5. Nodes (TBC)

Each node is a function `node(state) -> state`. It reads from state, do the required steps, log itself, and writes its result back.

In [ ]:
def extract_constraints(state):
    # (TBC): ask_json the request -> list of {type, value, is_hard}
    log('node', name='extract_constraints')
    state['constraints'] = [Constraint(type='raw', value=state['request'], is_hard=False, note='stub')]
    return state

def retrieve_info(state):
    # (TBC): pick tools based on constraints
    log('node', name='retrieve_info')
    state['info'] = {'attractions': search_attractions(limit=5)}
    return state

PLANNER_SYSTEM_PROMPT = """You are a careful New York travel itinerary planner.
Produce ONLY a JSON object with the following exact top-level keys:
{
  "destination": "New York",
  "days": [
    {"day_index": 1, "activities": [
      {"name": "venue name", "category": "category or null", "start_time": "HH:MM or null", "end_time": "HH:MM or null", "location": "borough or null", "price_usd": 0.0}
    ]}
  ],
  "total_cost_usd": null,
  "summary": "Short explanation including missing information or unmet requirements"
}
All day_index values are consecutive, starting at 1.
Use ONLY venue names provided in retrieved_info; never invent attractions, restaurants, hotels, ticket availability, or prices.
Treat hard constraints as mandatory whenever feasible. Prioritize soft constraints. In particular respect the requested trip length, budget, earliest start time, and maximum number of activities per day.
Use retrieved admission prices when available. price_usd represents the listed per-person admission price for that activity, not an invented total. For free venues use 0. For unavailable prices use null.
When the number of travellers is known, you may compute an estimated admission subtotal from listed per-person prices and traveller count. Set total_cost_usd to null if a complete trip total cannot be established; do not present admission-only totals as full trip totals. Explain excluded cost categories in summary.
Never claim a budget is satisfied when total cost is unknown. If retrieved venues are insufficient, return fewer activities or empty days rather than inventing venues. Do not fabricate travel times, opening hours, or availability. Use reasonable suggested time slots, clearly treated as proposals rather than verified opening times.
Return JSON only, no Markdown or commentary outside the JSON."""

def plan_itinerary(state):
    log('node', name='plan_itinerary')

    # Get retrieved attractions (Person A's original code)
    got = (state.get('info') or {}).get('attractions') or []

    # Convert retrieved attractions into Activity objects
    acts = [
        Activity(
            name=a['name'],
            category=a.get('category'),
            location=a.get('borough'),
            price_usd=(
                float(a['admission_price_usd'])
                if a.get('admission_price_usd') is not None
                and pd.notna(a['admission_price_usd'])
                else None
            )
        )
        for a in got
        if isinstance(a, dict) and a.get('name')
    ]

    # Get constraints
    constraints = [
        c.model_dump() if isinstance(c, Constraint) else c
        for c in state.get('constraints', [])
    ]

    # Get all retrieved information
    retrieved_info = state.get('info') or {}

    # Prepare information for GPT-4.1
    payload = {
        'request': state['request'],
        'constraints': constraints,
        'retrieved_info': retrieved_info
    }

    # Construct LLM messages
    messages = [
        {
            'role': 'system',
            'content': PLANNER_SYSTEM_PROMPT
        },
        {
            'role': 'user',
            'content': json.dumps(
                payload,
                ensure_ascii=False,
                default=str
            )
        }
    ]

    # Generate itinerary (maximum 3 model calls)
    for attempt in range(1, 4):
        try:
            data = ask_json(messages, tries=1)

            if not isinstance(data, dict):
                raise ValueError(
                    'Expected a JSON object'
                )

            # 8. Validate itinerary structure
            itinerary = Itinerary(**data)

            if not itinerary.days:
                raise ValueError('Itinerary must contain at least one day')

            if any(
                day.day_index != index
                for index, day in enumerate(
                    itinerary.days, start=1
                )
            ):
                raise ValueError('day_index must be consecutive starting from 1')

            # 9. Store generated itinerary in state
            state['draft'] = itinerary

            log(
                'planner_success',
                attempt=attempt,
                days=len(itinerary.days)
            )

            return state

        except (ValueError, TypeError) as exc:
            log(
                'planner_format_error',
                attempt=attempt,
                error=str(exc)[:300]
            )

            if attempt == 3:
                # used as a fallback when the LLM fails
                state['draft'] = Itinerary(
                    destination='New York',
                    days=[
                        Day(
                            day_index=1,
                            activities=acts[:3]
                        )
                    ],
                    summary='stub'
                )

                log('planner_fallback', reason='LLM failed after 3 attempts')

                return state

            messages.append({
                'role': 'user',
                'content': (
                    'Your previous response was invalid: '
                    + str(exc)[:500]
                    + '. Return valid JSON matching the '
                    'Itinerary schema. Do not invent venues.'
                )
            })

## 6. Graph + Run

Straight line for Baseline.

In [9]:
# Current Graph: only need straight line (list of nodes in order)
PIPELINE = [extract_constraints, retrieve_info, plan_itinerary]

# (TBC) Add verify/repair node later (useful for V1,V2,V3,V4)

# Make a new fresh State
def run(request):
    LOG.clear()
    log('start', request=request)
    state = State(request=request)
    for node in PIPELINE:
        state = node(state)
    state['final'] = state.get('draft') # baseline: no verification
    log('end')
    return state['final']

## 7. Testing

In [10]:
req = ('Plan a 3-day trip to New York for two with a $1500 budget. Vegetarian, '
       'relaxing, max 3 activities per day, visit a park, nothing before 10am.')
plan = run(req)
print(plan.model_dump_json(indent=2))

{
  "destination": "New York",
  "days": [
    {
      "day_index": 1,
      "activities": [
        {
          "name": "TIMES SQUARE PLAZA",
          "category": "landmark",
          "start_time": null,
          "end_time": null,
          "location": "Manhattan",
          "price_usd": null
        },
        {
          "name": "HARLEM RIVER PARK W 145 ST",
          "category": "park",
          "start_time": null,
          "end_time": null,
          "location": "Manhattan",
          "price_usd": null
        },
        {
          "name": "LUNT FONTANNE THEATRE",
          "category": "theater",
          "start_time": null,
          "end_time": null,
          "location": "Manhattan",
          "price_usd": null
        }
      ]
    }
  ],
  "total_cost_usd": null,
  "summary": "stub"
}


In [11]:
# see the log
for e in LOG:
    print(e)

{'kind': 'start', 't': 1791710642.9, 'request': 'Plan a 3-day trip to New York for two with a $1500 budget. Vegetarian, relaxing, max 3 activities per day, visit a park, nothing before 10am.'}
{'kind': 'node', 't': 1791710642.9, 'name': 'extract_constraints'}
{'kind': 'node', 't': 1791710642.9, 'name': 'retrieve_info'}
{'kind': 'tool', 't': 1791710642.901, 'name': 'search_attractions', 'n': 5}
{'kind': 'node', 't': 1791710642.901, 'name': 'plan_itinerary'}
{'kind': 'end', 't': 1791710642.901}
